In [26]:
import pandas as pd
import numpy as np

In [27]:
# 設定您的檔案路徑 (請確認這三個檔案在您的資料夾內)
file_esg = 'C:\\Users\\170134\\Desktop\\新增資料夾\\esg_data.csv'
file_return = 'C:\\Users\\170134\\Desktop\\新增資料夾\\return_data.csv'
# TEJPRO匯出時可選擇匯出產業分類欄位
# file_industry = 'industry_data.csv'

In [ ]:
# ==========================================
# 1. 讀取 ESG 資料
# ==========================================
print("正在讀取 ESG 資料...")
# encoding='cp950' 是關鍵，因為 TEJ Pro 預設是 Big5 編碼
# thousands=',' 用來處理數字中的逗號 (例如 1,000)
try:
    df_esg = pd.read_csv(file_esg, encoding='cp950')
except UnicodeDecodeError:
    df_esg = pd.read_csv(file_esg, encoding='utf-8') # 備用方案

# 重新命名欄位 (請根據您 CSV 裡的實際中文欄位名稱修改這裡的 key)
# 假設 TEJ 匯出的欄位名稱如下：
df_esg = df_esg.rename(columns={
    '證券代碼': 'Company',
    'TSE產業_名稱': 'Industry',
    'TESG評等公告日': 'Date',     # 防呆
    'TESG分數': 'Total_ESG', # 或是 'TESG評等'
    '環境構面分數': 'E_Score',
    '社會構面分數': 'S_Score',
    '公司治理構面分數': 'G_Score'
})

# 處理日期：轉成 datetime 格式，並抓出 "Year"
df_esg['Date'] = pd.to_datetime(df_esg['Date'].astype(str))
df_esg['Year'] = df_esg['Date'].dt.year


# --- (C) 核心邏輯：保留該年度最後一筆 ---
print(f"原始資料筆數: {len(df_esg)}")


# 1. 排序：先排公司，再排日期 (由舊到新)
# 這樣同一家公司同一年的資料，最下面那一筆一定是 "日期最晚" 的
df_esg = df_esg.sort_values(by=['Company', 'Date'], ascending=[True, True])

# 2. 去除重複：針對 'Company' 和 'Year' 這兩欄一樣的資料，只保留 'last' (最後一筆)
df_esg = df_esg.drop_duplicates(subset=['Company', 'Year'], keep='last')

print(f"篩選後筆數 (一年一筆): {len(df_esg)}")

# 只留需要的欄位
cols_to_keep = ['Company', 'Industry','Year', 'Total_ESG', 'E_Score', 'S_Score', 'G_Score']
# 檢查欄位是否存在，只選存在的
df_esg = df_esg[[c for c in cols_to_keep if c in df_esg.columns]]


正在讀取 ESG 資料...
原始資料筆數: 19069
篩選後筆數 (一年一筆): 9841


In [29]:
# ==========================================
# 2. 讀取 報酬率 資料
# ==========================================
print("正在讀取 報酬率 資料...")
try:
    df_ret = pd.read_csv(file_return, encoding='cp950')
except:
    df_ret = pd.read_csv(file_return, encoding='utf-8')

df_ret = df_ret.rename(columns={
    '證券代碼': 'Company',
    '年月日': 'Date',
    '近一年報酬率 %': 'Return' # 假設您直接下載算好的年報酬率 (%)
})

df_ret['Date'] = pd.to_datetime(df_ret['Date'].astype(str))
df_ret['Year'] = df_ret['Date'].dt.year

# --- (C) 核心邏輯：保留該年度最後一筆 ---
print(f"原始資料筆數: {len(df_ret)}")


# 1. 排序：先排公司，再排日期 (由舊到新)
# 這樣同一家公司同一年的資料，最下面那一筆一定是 "日期最晚" 的
df_ret = df_ret.sort_values(by=['Company', 'Date'], ascending=[True, True])

# 2. 去除重複：針對 'Company' 和 'Year' 這兩欄一樣的資料，只保留 'last' (最後一筆)
df_ret = df_ret.drop_duplicates(subset=['Company', 'Year'], keep='last')

print(f"篩選後筆數 (一年一筆): {len(df_ret)}")


# 確保報酬率是數字 (TEJ 有時候會有文字)
df_ret['Return'] = pd.to_numeric(df_ret['Return'], errors='coerce')

df_ret = df_ret[['Company', 'Year', 'Return']]

正在讀取 報酬率 資料...
原始資料筆數: 3856540
篩選後筆數 (一年一筆): 19333


In [30]:
# ==========================================
# 4. 資料大合併 (Merge)
# ==========================================
print("正在合併資料...")

# 公司代碼通常需要標準化 (有些是文字 '2330'，有些是數字 2330)
# 我們統一轉成字串
df_esg['Company'] = df_esg['Company'].astype(str).str.strip()
df_ret['Company'] = df_ret['Company'].astype(str).str.strip()

# 步驟 A: 合併 ESG + 報酬率 (Key: Company + Year)
df_final = pd.merge(df_ret, df_esg, on=['Company', 'Year'], how='inner')

# 步驟 B: 合併 產業 (Key: Company)
# df_final = pd.merge(df_final, df_ind, on='Company', how='left')

# 去除空值
df_final = df_final.dropna()

print("\n=== 資料處理完成！預覽前 5 筆 ===")
print(df_final.head())
print(f"\n總資料筆數: {len(df_final)}")

# 檢查一下有沒有成功讀到數據
if 'Total_ESG' in df_final.columns:
    print(f"ESG 分數平均值: {df_final['Total_ESG'].mean():.2f}")
    
# 匯出整理好的檔案，供下一步分析使用
# df_final.to_csv('cleaned_data_for_analysis.csv', index=False, encoding='utf-8-sig')
# print("已儲存為 'cleaned_data_for_analysis.csv'")

正在合併資料...

=== 資料處理完成！預覽前 5 筆 ===
   Company  Year   Return  Total_ESG  E_Score  S_Score  G_Score
0  1101 台泥  2022 -21.0490      61.79    58.03    76.94    55.33
1  1101 台泥  2023   4.8263      67.87    78.75    69.93    54.65
2  1101 台泥  2024  -6.3279      61.98    72.59    59.64    52.10
3  1101 台泥  2025 -23.9159      61.86    71.75    60.65    51.98
4  1102 亞泥  2022   0.3192      64.49    66.30    78.11    53.07

總資料筆數: 8688
ESG 分數平均值: 53.71


In [31]:
# 檢查一下有沒有成功讀到數據
if 'Total_ESG' in df_final.columns:
    print(f"ESG 分數平均值: {df_final['Total_ESG'].mean():.2f}")
    
# 匯出整理好的檔案，供下一步分析使用
df_final.to_csv('C:\\Users\\170134\\Desktop\\新增資料夾\\cleaned_data_for_analysis.csv', index=False, encoding='utf-8-sig')
print("已儲存為 'cleaned_data_for_analysis.csv'")

ESG 分數平均值: 53.71
已儲存為 'cleaned_data_for_analysis.csv'
